U ovoj svesci treniramo model zasnovan na **pretreniranoj ResNet18 mreži**. Koristićemo isti skup podataka i podelu kao i u prethodnim sveskama.

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

#### PUTANJE, PARAMETRI, DEVICE

In [ ]:
base_dir = Path("../data/tomatoleaf/tomato")

train_dir = base_dir / "train"
test_dir = base_dir / "val"

print("Train dir:", train_dir.resolve())
print("Test dir:", test_dir.resolve())
print("Train exists:", train_dir.exists())
print("Test exists:", test_dir.exists())

# ResNet18 je pretreniran na ImageNet skupu, pa je prirodno koristiti 224x224.
# Ako je treniranje presporo na CPU, može se probati IMG_SIZE = 128, ali je 224 standardnije za transfer learning.
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2
SEED = 42

best_model_path = Path("transfer_learning_resnet18_best.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#### TRANSFORMACIJE SLIKA

Koristimo iste ImageNet vrednosti za normalizaciju kao u prethodnim sveskama zato što se ResNet18 trenira nad ImageNet slikama sa baš ovom normalizacijom.

In [ ]:
def add_windows_long_path_prefix(path):
    """
    Na Windowsu neki fajlovi iz ovog dataset-a imaju jako duga imena.
    Ako je apsolutna putanja duža od Windows MAX_PATH ograničenja, open() može da prijavi
    FileNotFoundError iako fajl stvarno postoji. Prefiks \\?\ omogućava čitanje dugih putanja.
    Na Linuxu/macOS-u funkcija samo vraća običnu putanju.
    """
    path = Path(path).resolve()
    path_str = str(path)

    if os.name == "nt":
        if path_str.startswith("\\\\"):
            return "\\\\?\\UNC\\" + path_str.lstrip("\\")
        return "\\\\?\\" + path_str

    return path_str


def pil_loader_safe(path):
    """
    Loader za torchvision ImageFolder.
    Koristi Windows long-path prefiks i konvertuje sve slike u RGB.
    """
    with open(add_windows_long_path_prefix(path), "rb") as f:
        image = Image.open(f)
        return image.convert("RGB")


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

#### UČITAVANJE SKUPA PODATAKA

Da bismo imali augmentacije samo na treningu, pravimo dva `ImageFolder` objekta nad istim `train` folderom:
- jedan sa `train_transform`;
- drugi sa `eval_transform`.

Zatim koristimo iste indekse za train/validation podelu. Dakle, imamo dva `ImageFolder` objekta nad istim train_dir, ali sa različitim transformacijama.

In [ ]:
full_train_dataset_augmented = datasets.ImageFolder(
    root=str(train_dir),
    transform=train_transform,
    loader=pil_loader_safe
)

full_train_dataset_eval = datasets.ImageFolder(
    root=str(train_dir),
    transform=eval_transform,
    loader=pil_loader_safe
)

test_dataset = datasets.ImageFolder(
    root=str(test_dir),
    transform=eval_transform,
    loader=pil_loader_safe
)

class_names = full_train_dataset_augmented.classes
num_classes = len(class_names)

print("Klase:", class_names)
print("Broj klasa:", num_classes)
print("Broj slika u originalnom train skupu:", len(full_train_dataset_augmented))
print("Broj slika u test skupu:", len(test_dataset))

#### TRAIN/VALIDATION PODELA

In [ ]:
val_size = int(len(full_train_dataset_augmented) * VALIDATION_SPLIT)
train_size = len(full_train_dataset_augmented) - val_size

generator = torch.Generator().manual_seed(SEED)

indices = torch.randperm(len(full_train_dataset_augmented), generator=generator).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = torch.utils.data.Subset(full_train_dataset_augmented, train_indices)
val_dataset = torch.utils.data.Subset(full_train_dataset_eval, val_indices)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

#### DATALOADERI

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)